# Asteroid Orbital Visual — Automated Pipeline
Run **Cell → Run All** to execute the full pipeline end-to-end:
1. Fetch raw NASA CNEOS data from GitHub and flatten on the fly
2. Verify Node.js / npm
3. Install `powerbi-visuals-tools` globally (skipped if already installed)
4. Install visual npm dependencies
5. Install the pbiviz developer certificate
6. Start the pbiviz dev server (background)
7. Launch Power BI Desktop
8. Print field-mapping checklist + Power BI Web connector URL
9. Preview and profile the flattened data — all from GitHub

In [ ]:
import subprocess, sys, os, time, threading, glob
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd())
VISUAL_DIR   = NOTEBOOK_DIR / 'asteroid-orbital-visual'
FLAT_CSV     = NOTEBOOK_DIR / 'asteroids_flat.csv'

GITHUB_REPO     = 'jdstigma/nasa-asteroids'
GITHUB_BRANCH   = 'main'
GITHUB_RAW_BASE = f'https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_BRANCH}'
RAW_CSV_URL     = f'{GITHUB_RAW_BASE}/asteroids_data.csv'
FLAT_CSV_URL    = f'{GITHUB_RAW_BASE}/asteroids_flat.csv'

def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    return r.stdout.strip() or r.stderr.strip(), r.returncode

def ok(msg):   print(f'  ✅ {msg}')
def err(msg):  print(f'  ❌ {msg}')
def info(msg): print(f'  ℹ  {msg}')

print('Config')
print(f'  GitHub repo  : {GITHUB_REPO}')
print(f'  Raw CSV URL  : {RAW_CSV_URL}')
print(f'  Flat CSV URL : {FLAT_CSV_URL}')

## Step 1 — Fetch from GitHub and flatten

In [ ]:
import requests, io, csv, ast, json

print('Step 1 — Fetching asteroids_data.csv from GitHub...')

PARENT_COLS = [
    'id','neo_id','name','short_name','designation',
    'magnitude','potentially_hazardous',
    'diameter_min_m','diameter_max_m',
    'eccentricity','semi_major_axis','inclination',
    'ascending_node_longitude','orbital_period',
    'perihelion_distance','perihelion_argument','aphelion_distance',
    'mean_anomaly','mean_motion',
    'orbit_class_type','orbit_class_desc',
    'first_observation_date','last_observation_date','data_arc_days',
    'orbit_uncertainty','min_orbit_intersection','jupiter_tisserand',
]
OUTPUT_COLS = PARENT_COLS + [
    'close_approach_date','close_approach_date_full','epoch_ms',
    'velocity_km_s','velocity_km_h',
    'miss_distance_au','miss_distance_lunar','miss_distance_km',
    'orbiting_body',
]

def parse_approach(raw):
    if not raw or raw.strip() in ('', '[]'): return []
    try: return ast.literal_eval(raw)
    except Exception: pass
    try:
        fixed = raw.replace("'", '"').replace('None','null').replace('True','true').replace('False','false')
        return json.loads(fixed)
    except Exception: return []

# Stream raw CSV from GitHub
resp = requests.get(RAW_CSV_URL, stream=True, timeout=120)
resp.raise_for_status()
total = int(resp.headers.get('content-length', 0))
chunks = []
downloaded = 0
for chunk in resp.iter_content(chunk_size=256 * 1024):
    chunks.append(chunk)
    downloaded += len(chunk)
    if total:
        print(f'    {downloaded/1024/1024:.1f} / {total/1024/1024:.1f} MB  ({downloaded/total*100:.0f}%)', end='\r')
print()
raw_text = b''.join(chunks).decode('utf-8')
ok(f'Downloaded {downloaded/1024/1024:.1f} MB')

# Flatten in memory and write to FLAT_CSV
csv.field_size_limit(10_000_000)
reader = csv.DictReader(io.StringIO(raw_text))
rows_written = 0

with open(FLAT_CSV, 'w', newline='', encoding='utf-8') as fout:
    writer = csv.DictWriter(fout, fieldnames=OUTPUT_COLS, extrasaction='ignore')
    writer.writeheader()
    for raw_row in reader:
        parent = {col: raw_row.get(col, '') for col in PARENT_COLS}
        approaches = parse_approach(raw_row.get('close_approach_data', ''))
        if not approaches:
            writer.writerow({**parent, **{c: '' for c in OUTPUT_COLS if c not in PARENT_COLS}})
            continue
        for appr in approaches:
            vel  = appr.get('relative_velocity', {})
            dist = appr.get('miss_distance', {})
            writer.writerow({
                **parent,
                'close_approach_date':      appr.get('close_approach_date', ''),
                'close_approach_date_full': appr.get('close_approach_date_full', ''),
                'epoch_ms':                 appr.get('epoch_date_close_approach', ''),
                'velocity_km_s':            vel.get('kilometers_per_second', ''),
                'velocity_km_h':            vel.get('kilometers_per_hour', ''),
                'miss_distance_au':         dist.get('astronomical', ''),
                'miss_distance_lunar':      dist.get('lunar', ''),
                'miss_distance_km':         dist.get('kilometers', ''),
                'orbiting_body':            appr.get('orbiting_body', ''),
            })
            rows_written += 1

ok(f'Flattened → asteroids_flat.csv  ({rows_written:,} rows, {FLAT_CSV.stat().st_size/1024/1024:.1f} MB)')

## Step 2 — Verify Node.js & npm

In [ ]:
print('Step 2 — Node.js / npm')
node_ver, node_rc = run('node --version')
npm_ver,  npm_rc  = run('npm --version')
if node_rc != 0 or npm_rc != 0:
    err('Node.js not found. Install from https://nodejs.org then restart the kernel.')
    raise EnvironmentError
ok(f'Node {node_ver}  |  npm {npm_ver}')

## Step 3 — Install powerbi-visuals-tools globally

In [ ]:
print('Step 3 — pbiviz CLI')
ver, rc = run('pbiviz --version')
if rc == 0:
    ok(f'pbiviz {ver} already installed')
else:
    info('Installing powerbi-visuals-tools globally...')
    out, rc = run('npm install -g powerbi-visuals-tools')
    if rc != 0: err(out); raise RuntimeError
    ver, _ = run('pbiviz --version')
    ok(f'pbiviz {ver} installed')

## Step 4 — Install visual npm dependencies

In [ ]:
print('Step 4 — npm install')
node_modules = VISUAL_DIR / 'node_modules'
if node_modules.exists():
    ok('node_modules already present — skipping')
else:
    info('Running npm install...')
    out, rc = run('npm install', cwd=str(VISUAL_DIR))
    if rc != 0: err(out); raise RuntimeError
    ok('Dependencies installed')

## Step 5 — Developer certificate

In [ ]:
print('Step 5 — Developer certificate')
cert_path = Path.home() / 'pbiviz-certs' / 'PowerBICustomVisualTest_public.pfx'
if cert_path.exists():
    ok('Certificate already exists')
else:
    info('Generating certificate...')
    out, rc = run('pbiviz install-cert', cwd=str(VISUAL_DIR))
    print(out)
    if cert_path.exists():
        ok('Certificate generated')
        passphrase = [l for l in out.splitlines() if 'Passphrase' in l]
        pw = passphrase[0].split()[-1] if passphrase else 'SEE ABOVE'
        print()
        print('  ⚠  ACTION REQUIRED — run this in an Admin PowerShell to trust the cert:')
        print(f"  Import-PfxCertificate -FilePath '{cert_path}' -CertStoreLocation Cert:\\CurrentUser\\Root -Password (ConvertTo-SecureString '{pw}' -Force -AsPlainText)")
    else:
        err('Certificate generation failed — check output above')

## Step 6 — Start the pbiviz dev server

In [ ]:
print('Step 6 — pbiviz dev server')

server_proc = subprocess.Popen(
    'pbiviz start', shell=True, cwd=str(VISUAL_DIR),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

server_lines = []
def _stream():
    for line in server_proc.stdout:
        server_lines.append(line.rstrip())
threading.Thread(target=_stream, daemon=True).start()
time.sleep(8)

errors    = [l for l in server_lines if 'error' in l.lower() and 'deprecation' not in l.lower()]
listening = any('8080' in l or 'listening' in l.lower() for l in server_lines)
compiled  = any('compiled successfully' in l for l in server_lines)

if errors and not compiled:
    for l in server_lines[-20:]: print(f'    {l}')
    err('Errors detected')
elif listening and compiled:
    ok('Dev server running on https://localhost:8080  |  webpack compiled successfully')
else:
    info('Server still starting...')
    for l in server_lines[-10:]: print(f'    {l}')

def kill_server():
    server_proc.terminate()
    print('Dev server stopped.')

## Step 7 — Launch Power BI Desktop

In [ ]:
print('Step 7 — Launch Power BI Desktop')

pbi_candidates = [
    r'C:\Program Files\Microsoft Power BI Desktop\bin\PBIDesktop.exe',
    r'C:\Program Files (x86)\Microsoft Power BI Desktop\bin\PBIDesktop.exe',
    str(Path.home() / 'AppData/Local/Microsoft/WindowsApps/PBIDesktop.exe'),
]
pbi_candidates += glob.glob(
    str(Path.home() / 'AppData/Local/Microsoft/WindowsApps/**/PBIDesktop.exe'), recursive=True
)
pbi_exe  = next((p for p in pbi_candidates if Path(p).exists()), None)
pbix_files = list(NOTEBOOK_DIR.glob('*.pbix'))
pbix_arg   = str(pbix_files[0]) if pbix_files else None

if pbi_exe:
    subprocess.Popen([pbi_exe] + ([pbix_arg] if pbix_arg else []))
    ok(f'Launched Power BI Desktop — {Path(pbix_arg).name if pbix_arg else "blank project"}')
else:
    err('Power BI Desktop not found.')
    info('Install from https://powerbi.microsoft.com/desktop or the Microsoft Store, then re-run this cell.')

## Step 8 — Field mapping checklist & Web connector URL
In Power BI Desktop: **Home → Get data → Web** → paste the URL below.

In [ ]:
import pandas as pd

print('Step 8 — Power BI connection')
print()
print('  Power BI Web connector URL:')
print(f'  {FLAT_CSV_URL}')
print()
print('  Field mapping — drag into the Developer Visual "Data Fields" bucket:')
print('  ' + '-' * 58)
for col, role in [
    ('name',                  'Asteroid identity'),
    ('short_name',            'Label in tooltips'),
    ('potentially_hazardous', 'Red color + glow pulse'),
    ('diameter_max_m',        'Dot size'),
    ('semi_major_axis',       'Orbit radius (AU)'),
    ('eccentricity',          'Orbit shape'),
    ('perihelion_argument',   'Orbit orientation'),
    ('miss_distance_au',      'Closest approach distance (tooltip)'),
    ('close_approach_date',   'Approach date (tooltip)'),
    ('velocity_km_s',         'Approach speed (tooltip)'),
    ('orbiting_body',         'Which planet approached'),
    ('magnitude',             'Brightness (tooltip)'),
    ('orbit_class_type',      'Class label (tooltip)'),
]:
    print(f'  {col:<30} →  {role}')
print()
print('  Formatting pane:')
print('    Show All 8 Planets    — toggle outer planets on/off')
print('    Animation Speed       — days per frame (default 5)')
print('    Hazardous Only        — show only red asteroids')

## Step 9 — Data profile (all from GitHub)

In [ ]:
import matplotlib.pyplot as plt

print('Step 9 — Loading asteroids_flat.csv from GitHub...')
df = pd.read_csv(FLAT_CSV_URL, low_memory=False)
print(f'  Loaded {len(df):,} rows')

df['miss_distance_au']    = pd.to_numeric(df['miss_distance_au'],    errors='coerce')
df['velocity_km_s']       = pd.to_numeric(df['velocity_km_s'],       errors='coerce')
df['diameter_max_m']      = pd.to_numeric(df['diameter_max_m'],      errors='coerce')
df['close_approach_date'] = pd.to_datetime(df['close_approach_date'], errors='coerce')

print(f'  Asteroids     : {df["name"].nunique():,} unique')
print(f'  Hazardous     : {(df["potentially_hazardous"]=="True").sum():,} approaches')
print(f'  Date range    : {df["close_approach_date"].min().date()} → {df["close_approach_date"].max().date()}')
print(f'  Orbiting body : {df["orbiting_body"].value_counts().to_dict()}')
print()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('NASA CNEOS Asteroid Close Approaches', fontsize=13, fontweight='bold')

axes[0].hist(df[df['miss_distance_au'] < 0.5]['miss_distance_au'].dropna(),
             bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set(title='Miss Distance (< 0.5 AU)', xlabel='Miss Distance (AU)', ylabel='Approach count')

df['decade'] = df['close_approach_date'].dt.year // 10 * 10
dc = df['decade'].value_counts().sort_index()
axes[1].bar(dc.index.astype(str), dc.values, color='darkorange', edgecolor='white', linewidth=0.3)
axes[1].set(title='Close Approaches by Decade', xlabel='Decade', ylabel='Count')
axes[1].tick_params(axis='x', rotation=45)

axes[2].hist(df[df['potentially_hazardous']=='False']['velocity_km_s'].dropna(),
             bins=40, alpha=0.6, color='gray', label='Non-hazardous', edgecolor='white', linewidth=0.2)
axes[2].hist(df[df['potentially_hazardous']=='True']['velocity_km_s'].dropna(),
             bins=40, alpha=0.8, color='crimson', label='Hazardous', edgecolor='white', linewidth=0.2)
axes[2].set(title='Velocity by Hazard Status', xlabel='Velocity (km/s)', ylabel='Count')
axes[2].legend()

plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / 'assets' / 'data_profile.png', dpi=120, bbox_inches='tight')
plt.show()
print('  Saved → assets/data_profile.png')

In [ ]:
# Top 10 closest Earth approaches
closest = (df[df['orbiting_body']=='Earth']
           .sort_values('miss_distance_au')
           [['name','close_approach_date','miss_distance_au','velocity_km_s','potentially_hazardous']]
           .drop_duplicates('name').head(10).reset_index(drop=True))
closest.index += 1
print('Top 10 closest Earth approaches (one per asteroid):')
closest

In [ ]:
# Stop the dev server when done
# kill_server()